In [1]:
# Cell 0 — Drive Setup and Imports
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, glob, json, warnings, urllib.request
import numpy as np
import pandas as pd
import joblib
import stumpy

from numpy.linalg import pinv
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

warnings.filterwarnings('ignore')
RANDOMSTATE = 42
np.random.seed(RANDOMSTATE)

try:
    from numba import cuda
    USEGPU = cuda.is_available()
    print('GPU', 'Available' if USEGPU else 'Not available — CPU only')
except Exception:
    USEGPU = False
    print('GPU check failed — CPU only')

# Unified Drive paths
RUNID = 'ensemble_run_001'
DRIVEROOT = '/content/drive/MyDrive/tsad_ensemble_runs'
NOTEBOOKTAG = 'matrix_profile'

RUNDIR = os.path.join(DRIVEROOT, RUNID, NOTEBOOKTAG)
ARTIFACTDIR = os.path.join(RUNDIR, 'artifacts')
PREDICTIONSDIR = os.path.join(RUNDIR, 'predictions')
CACHEDIR = os.path.join(DRIVEROOT, '_cache')

for d in [ARTIFACTDIR, PREDICTIONSDIR, CACHEDIR]:
    os.makedirs(d, exist_ok=True)

print('RUNDIR       :', RUNDIR)
print('ARTIFACTDIR  :', ARTIFACTDIR)
print('PREDICTIONSDIR:', PREDICTIONSDIR)


Mounted at /content/drive
GPU Not available — CPU only
RUNDIR       : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile
ARTIFACTDIR  : /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/artifacts
PREDICTIONSDIR: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions


In [2]:
# Cell 1 — Dataset Paths

MYDRIVE = '/content/drive/MyDrive'
CREDITCARDPATH = os.path.join(MYDRIVE, 'creditcard.csv')

NAB_CANDIDATES = [
    os.path.join(MYDRIVE, 'NAB Dataset'),
    os.path.join(MYDRIVE, 'NAB'),
    os.path.join(MYDRIVE, 'datasets', 'NAB Dataset'),
    os.path.join(MYDRIVE, 'datasets', 'NAB'),
]

def resolve_nab_root(candidates):
    for c in candidates:
        if os.path.isdir(c):
            csvs = glob.glob(os.path.join(c, '**', '*.csv'), recursive=True)
            if len(csvs) > 0:
                return c
    return None

NABROOT = resolve_nab_root(NAB_CANDIDATES)

# NAB labels
NABLABELSLOCAL = os.path.join(CACHEDIR, 'nab_combined_windows.json')
NABLABELSURL = 'https://raw.githubusercontent.com/numenta/NAB/master/labels/combined_windows.json'
if not os.path.exists(NABLABELSLOCAL):
    print('Downloading NAB labels...')
    urllib.request.urlretrieve(NABLABELSURL, NABLABELSLOCAL)

with open(NABLABELSLOCAL) as f:
    NABWINDOWSMAP = json.load(f)

assert os.path.exists(CREDITCARDPATH), f'creditcard.csv not found at {CREDITCARDPATH}'
assert NABROOT is not None, 'NAB root not found in Drive.'

nab_csvs = [f for f in glob.glob(os.path.join(NABROOT, '**', '*.csv'), recursive=True) if 'README' not in f]
print(f'CREDITCARDPATH: {CREDITCARDPATH}')
print(f'NABROOT       : {NABROOT}')
print(f'NAB CSV count : {len(nab_csvs)}')
print(f'NAB labels    : {len(NABWINDOWSMAP)} entries')


CREDITCARDPATH: /content/drive/MyDrive/creditcard.csv
NABROOT       : /content/drive/MyDrive/NAB Dataset
NAB CSV count : 58
NAB labels    : 58 entries


In [3]:
# Cell 2 — Shared Utilities

def robust_z(x, eps=1e-9, clip=50.0):
    x = np.asarray(x, dtype=np.float64)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    return np.clip((x - med) / max(1.4826 * mad, eps), -clip, clip)

def keep_runs(y, min_len=3):
    y = np.asarray(y, dtype=np.int8).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] == 1:
            j = i
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_len:
                y[i:j] = 0
            i = j
        else:
            i += 1
    return y

def point_adjust(y_true, y_pred):
    yt = np.asarray(y_true, dtype=np.int8)
    yp = np.asarray(y_pred, dtype=np.int8).copy()
    n = len(yt)
    i = 0
    while i < n:
        if yt[i] == 1:
            j = i
            while j < n and yt[j] == 1:
                j += 1
            if yp[i:j].any():
                yp[i:j] = 1
            i = j
        else:
            i += 1
    return yp

def rolling_max_1d(x, w):
    x = np.asarray(x, dtype=np.float64)
    if w <= 1:
        return x
    return pd.Series(x).rolling(w, min_periods=1).max().values

def best_strict_f1_threshold(y, s, ngrid=300, qlo=0.50, qhi=0.999, minrun=3):
    y = np.asarray(y, dtype=int)
    s = np.asarray(s, dtype=float)
    if y.sum() == 0:
        return float(np.max(s)) if len(s) else 0.0, 0.0
    qs = np.linspace(qlo, qhi, ngrid)
    thr_list = np.unique(np.quantile(s, qs))
    best_t, best_f = float(thr_list[-1]), -1.0
    for t in thr_list:
        p = keep_runs((s >= t).astype(int), min_len=minrun)
        f = f1_score(y, p, zero_division=0)
        if f > best_f:
            best_f, best_t = float(f), float(t)
    return best_t, best_f

def compute_metrics(y, pred, scores=None, prefix=''):
    m = {
        f'{prefix}precision': float(precision_score(y, pred, zero_division=0)),
        f'{prefix}recall': float(recall_score(y, pred, zero_division=0)),
        f'{prefix}f1': float(f1_score(y, pred, zero_division=0)),
    }
    if scores is not None and len(np.unique(y)) == 2:
        m[f'{prefix}rocauc'] = float(roc_auc_score(y, scores))
        m[f'{prefix}prauc'] = float(average_precision_score(y, scores))
    else:
        m[f'{prefix}rocauc'] = float('nan')
        m[f'{prefix}prauc'] = float('nan')
    return m

print('Utilities ready.')


Utilities ready.


In [4]:
# DEBUG — Verify utility functions work correctly
print('=== UTILITY FUNCTION VERIFICATION ===')

# Test keep_runs
test_y = np.array([0,0,1,1,0,0,1,0,1,1,1,1,0], dtype=np.int8)
result = keep_runs(test_y, min_len=3)
print(f'keep_runs input : {test_y.tolist()}')
print(f'keep_runs output: {result.tolist()}')
assert result[2:4].sum() == 0, 'Short run of 2 should be removed'
assert result[8:12].sum() == 4, 'Run of 4 should be kept'
print('keep_runs: PASS')

# Test best_strict_f1_threshold with known data
test_ytrue = np.array([0]*90 + [1]*10)
test_scores = np.array([0.1]*85 + [0.5]*5 + [0.8]*7 + [0.3]*3)
thr, f1 = best_strict_f1_threshold(test_ytrue, test_scores, ngrid=50, minrun=1)
print(f'best_strict_f1_threshold: thr={thr:.4f}, f1={f1:.4f}')
assert f1 > 0.0, 'Should find nonzero F1'
print('best_strict_f1_threshold: PASS (no self error, no minlen error)')

# Test point_adjust
pa = point_adjust(test_ytrue, (test_scores > 0.7).astype(np.int8))
print(f'point_adjust: detected {pa.sum()} vs true {test_ytrue.sum()}')
print('point_adjust: PASS')
print()
print('All utility functions verified.')


=== UTILITY FUNCTION VERIFICATION ===
keep_runs input : [0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0]
keep_runs output: [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0]
keep_runs: PASS
best_strict_f1_threshold: thr=0.7556, f1=0.8235
best_strict_f1_threshold: PASS (no self error, no minlen error)
point_adjust: detected 10 vs true 10
point_adjust: PASS

All utility functions verified.


In [5]:
# Cell 3 — Credit Card MP Agent

class CreditCardMPAgent:
    TOPK = 29
    OUTLIERFRAC = 0.005
    RANDOMSTATE = 42

    def __init__(self):
        self.topfeats = None
        self.mu = None
        self.covinv = None
        self.threshold = None
        self.featurecols = None
        self.testindices = None

    def load(self):
        df = pd.read_csv(CREDITCARDPATH).dropna()
        df = df.sort_values('Time').reset_index(drop=True)
        df['Amountlog'] = np.log1p(df['Amount'].astype(float))
        featurecols = [f'V{i}' for i in range(1, 29)] + ['Amountlog']
        X = df[featurecols].values.astype(np.float64)
        y = df['Class'].astype(int).values
        self.featurecols = featurecols
        return X, y

    def score(self, X):
        d = X - self.mu
        s = np.einsum('ij,jk,ik->i', d, self.covinv, d)
        s = np.sqrt(np.clip(np.nan_to_num(s, nan=0.0, posinf=0.0, neginf=0.0), 0.0, None))
        return s.astype(np.float64)

    def fit(self):
        Xall, yall = self.load()
        Xnormal = Xall[yall == 0]
        print(f'CC-MP rows {len(Xall):,} frauds {int(yall.sum())} rate {100*yall.mean():.4f}%')

        self.topfeats = np.arange(Xnormal.shape[1])[:self.TOPK]
        Xn = Xnormal[:, self.topfeats]

        # Robust fit: compute rough Mahalanobis, trim outliers, refit
        mu_rough = Xn.mean(axis=0)
        cov_rough = np.cov(Xn.T) + 1e-6 * np.eye(Xn.shape[1])
        cinv_rough = pinv(cov_rough)
        d_rough = Xn - mu_rough
        s_rough = np.sqrt(np.clip(np.einsum('ij,jk,ik->i', d_rough, cinv_rough, d_rough), 0.0, None))

        cutoff = np.percentile(s_rough, (1 - self.OUTLIERFRAC) * 100)
        Xclean = Xn[s_rough <= cutoff]
        print(f'CC-MP robust fit: kept {len(Xclean):,}/{len(Xn):,} normal rows')

        self.mu = Xclean.mean(axis=0)
        cov = np.cov(Xclean.T) + 1e-6 * np.eye(Xclean.shape[1])
        self.covinv = pinv(cov)
        return self

    def evaluate(self):
        Xall, yall = self.load()
        scores = self.score(Xall[:, self.topfeats])

        idx = np.arange(len(yall), dtype=np.int64)
        # Same split as Memto and IF: 80/20 with stratification
        idxtv, idxtest = train_test_split(
            idx, test_size=0.20, stratify=yall, random_state=self.RANDOMSTATE
        )
        # Further split tune into fit/val for threshold tuning
        idxfit, idxval = train_test_split(
            idxtv, test_size=0.25, stratify=yall[idxtv], random_state=self.RANDOMSTATE
        )

        # Threshold from PR-curve on validation
        prec, rec, thr = precision_recall_curve(yall[idxval], scores[idxval])
        f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
        if len(thr) == 0:
            self.threshold = float(np.percentile(scores[idxval], 99.5))
        else:
            self.threshold = float(thr[int(np.nanargmax(f1s))])

        ypredtest = (scores[idxtest] >= self.threshold).astype(np.int8)
        self.testindices = idxtest.astype(np.int64).copy()

        strict = compute_metrics(yall[idxtest], ypredtest, scores[idxtest])

        results = {
            'dataset': 'creditcard',
            'protocol': 'strict point-wise holdout (20% test)',
            **strict,
            'threshold': float(self.threshold),
            'n_anomalies_true': int(yall[idxtest].sum()),
            'n_anomalies_pred': int(ypredtest.sum()),
        }

        return (
            results,
            scores[idxtest].astype(np.float32),
            yall[idxtest].astype(np.int8),
            ypredtest.astype(np.int8),
        )

    def save_artifact(self):
        path = os.path.join(ARTIFACTDIR, 'mpcreditcard.joblib')
        joblib.dump({
            'topfeats': self.topfeats,
            'mu': self.mu,
            'covinv': self.covinv,
            'threshold': self.threshold,
            'featurecols': self.featurecols,
            'testindices': self.testindices,
        }, path)
        print('Saved artifact ->', path)

ccmpagent = CreditCardMPAgent()
print('CC-MP fitting...')
ccmpagent.fit()

print('CC-MP evaluating...')
ccmpresults, ccmpscores, ccmpytrue, ccmppred = ccmpagent.evaluate()
ccmpagent.save_artifact()

print()
print('=' * 60)
print('CREDIT CARD MATRIX PROFILE RESULTS')
print('=' * 60)
for k, v in ccmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')


CC-MP fitting...
CC-MP rows 284,807 frauds 492 rate 0.1727%
CC-MP robust fit: kept 282,893/284,315 normal rows
CC-MP evaluating...
Saved artifact -> /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/artifacts/mpcreditcard.joblib

CREDIT CARD MATRIX PROFILE RESULTS
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.821053
  recall                   : 0.795918
  f1                       : 0.808290
  rocauc                   : 0.964591
  prauc                    : 0.695749
  threshold                : 302.642458
  n_anomalies_true         : 98
  n_anomalies_pred         : 95


In [6]:
# DEBUG — CC Matrix Profile diagnostics
print('=== CC MATRIX PROFILE DIAGNOSTICS ===')
print(f'Threshold: {ccmpagent.threshold:.4f}')
print(f'Top features used: {ccmpagent.topfeats[:5]}...')
print(f'Covariance inverse shape: {ccmpagent.covinv.shape}')
print(f'Test set: {len(ccmpytrue)} samples, {int(ccmpytrue.sum())} frauds')
print(f'Predictions: {int(ccmppred.sum())} positive')
print(f'Score range: [{ccmpscores.min():.4f}, {ccmpscores.max():.4f}]')
print(f'Score at threshold percentile: {(ccmpscores < ccmpagent.threshold).mean()*100:.2f}%')
# Check FP/FN breakdown
fp = int(((ccmppred == 1) & (ccmpytrue == 0)).sum())
fn = int(((ccmppred == 0) & (ccmpytrue == 1)).sum())
tp = int(((ccmppred == 1) & (ccmpytrue == 1)).sum())
tn = int(((ccmppred == 0) & (ccmpytrue == 0)).sum())
print(f'Confusion: TP={tp} FP={fp} FN={fn} TN={tn}')
print(f'Precision={tp/(tp+fp+1e-9):.4f} Recall={tp/(tp+fn+1e-9):.4f}')


=== CC MATRIX PROFILE DIAGNOSTICS ===
Threshold: 302.6425
Top features used: [0 1 2 3 4]...
Covariance inverse shape: (29, 29)
Test set: 56962 samples, 98 frauds
Predictions: 95 positive
Score range: [2.2483, 1450.6675]
Score at threshold percentile: 99.83%
Confusion: TP=78 FP=17 FN=20 TN=56847
Precision=0.8211 Recall=0.7959


In [7]:
# Cell 6 — Export for Coordinator (Credit Card only)

exported = {}
summaryrows = []

# --- Credit Card ---
ccbundle = {
    'dataset': 'creditcard',
    'model': 'matrixprofile',
    'protocol': 'strict point-wise holdout',
    'entities': {
        'creditcard': {
            'entityid': 'creditcard',
            'scoresfull': ccmpscores,
            'yfull': ccmpytrue,
            'predfull': ccmppred,
            'rowid': np.arange(len(ccmpytrue), dtype=np.int64),
            'originalrowid': ccmpagent.testindices.astype(np.int64),
            'threshold': float(ccmpagent.threshold),
        }
    }
}

ccpath = os.path.join(PREDICTIONSDIR, 'mpcreditcardstrictpointwise.joblib')
joblib.dump(ccbundle, ccpath)
exported['creditcard'] = ccpath
summaryrows.append({'dataset': 'creditcard', **{k: v for k, v in ccmpresults.items() if k != 'dataset'}})
print('Saved', ccpath)

# --- Summary ---
summary = pd.DataFrame(summaryrows)
summarypath = os.path.join(PREDICTIONSDIR, 'mpstrictsummary.csv')
summary.to_csv(summarypath, index=False)
print('Saved', summarypath)
display(summary)

# --- Manifest ---
manifest = {
    'runid': RUNID,
    'driveroot': DRIVEROOT,
    'notebooktag': NOTEBOOKTAG,
    'modelfamily': 'matrixprofile',
    'exportprotocol': 'ensembleexportv2',
    'artifactsdir': ARTIFACTDIR,
    'predictionsdir': PREDICTIONSDIR,
    'exports': exported,
    'summarycsv': summarypath,
    'creditcard_originalrowid_included': True,
    'datasets': ['creditcard'],
}

manifestpath = os.path.join(PREDICTIONSDIR, 'mpmanifest.json')
with open(manifestpath, 'w') as f:
    json.dump(manifest, f, indent=2)
print('Saved', manifestpath)

print()
print('Coordinator-ready files in:', PREDICTIONSDIR)
print('Done.')

Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpcreditcardstrictpointwise.joblib
Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpstrictsummary.csv


,dataset,protocol,precision,recall,f1,rocauc,prauc,threshold,n_anomalies_true,n_anomalies_pred
0,creditcard,strict point-wise holdout (20% test),0.821053,0.795918,0.80829,0.964591,0.695749,302.642458,98,95


Saved /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions/mpmanifest.json

Coordinator-ready files in: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions
Done.


In [8]:
# Cell 7 — Final Summary

print('=' * 70)
print('MATRIX PROFILE FINAL SUMMARY')
print('=' * 70)

print()
print('CREDIT CARD:')
for k, v in ccmpresults.items():
    print(f'  {k:25s}: {v:.6f}' if isinstance(v, float) else f'  {k:25s}: {v}')

print()
print('All exports saved to:', PREDICTIONSDIR)
print('Done.')

MATRIX PROFILE FINAL SUMMARY

CREDIT CARD:
  dataset                  : creditcard
  protocol                 : strict point-wise holdout (20% test)
  precision                : 0.821053
  recall                   : 0.795918
  f1                       : 0.808290
  rocauc                   : 0.964591
  prauc                    : 0.695749
  threshold                : 302.642458
  n_anomalies_true         : 98
  n_anomalies_pred         : 95

All exports saved to: /content/drive/MyDrive/tsad_ensemble_runs/ensemble_run_001/matrix_profile/predictions
Done.
